# Download Studies Metadata

**Step 1 of 3** in the BioData Catalyst harmonization workflow:

1. **Download Studies Metadata** (this notebook) — fetch preharmonized PFBs from BDC and
   extract the dbGaP data dictionary and variable report XML files.
2. [Generate Preliminary Harmonization Mappings](./harmonize_studies.ipynb) — map each study
   variable to ranked candidate slots in the target schema.
3. [Review Harmonization Suggestions](./review_harmonization_suggestions.ipynb) — accept or
   skip candidates one variable at a time.

Where metadata files are bundled inside archives, they are extracted automatically. Studies
whose metadata cannot be found are reported to `outputs/` for further investigation.

> **Before you start:** this downloads whole study PFBs, which can be several GB per study and
> take a while. Files already on disk are skipped, so the notebook is safe to re-run and will
> pick up where it left off.

# Setup

## Libraries

Install with `pip install ai-harmonization gen3`.

In [ ]:
# Uncomment to install libraries used in this notebook
#!pip -q install ai-harmonization gen3

In [ ]:
import json
import os

import requests

from gen3.auth import Gen3Auth
from gen3.metadata import Gen3Metadata
from gen3.file import Gen3File

from ai_harmonization.gen3_utils import (
    get_active_preharmonized_studies,
    select_studies,
    download_pfb_for_study,
    convert_pfb_to_tsv,
    download_study_metadata,
    check_missing_metadata,
)

## Configuration

### Gen3 credentials

This notebook queries the BioData Catalyst metadata endpoint and downloads controlled-access
files, so it needs a Gen3 API key.

1. Sign in to [BioData Catalyst](https://gen3.biodatacatalyst.nhlbi.nih.gov) with the account
   that has dbGaP access to the studies you want.
2. Open **Profile → Create API key**, then **Download json**.
3. Save it as `gen3-credentials/bdc.json` next to this notebook (that directory is gitignored).

The key expires, so re-download it if authentication starts failing.

In [ ]:
# Path to credentials. Change as needed
credentials_path = './gen3-credentials/bdc.json'

# Commons URL. Change as needed
commons_url = 'https://gen3.biodatacatalyst.nhlbi.nih.gov'

### Which studies to download

These notebooks all work through one example study, `phs000704.v1.p1.c1`. Add more IDs to
`selected_study_ids` to process a batch.

`download_mode` decides which selector is used:

| Mode | Downloads |
|------|-----------|
| `'selected'` | only the study IDs in `selected_study_ids` |
| `'max'` | the first `max_studies_to_process` studies in the catalog |
| `'all'` | every active study with preharmonized data (hundreds of GB) |

Not every study can be processed. Some are archived (`doi_tombstone=True`), some publish no
`data_dict.xml` or `var_report.xml`, and some sit in buckets whose access groups are not
configured yet — those are reported by the last cell of this notebook.

In [ ]:
download_mode = 'selected'

# Used when download_mode = 'selected'
selected_study_ids = [
    'phs000704.v1.p1.c1',
]

# Used when download_mode = 'max'
max_studies_to_process = 5

## Inputs / Outputs

- `inputs/studies/{study_id}/` — downloaded PFB, its TSV manifests, and the extracted
  `metadata/` XMLs. This is what the next notebook reads.
- `outputs/` — reports about this run: the index of selected studies and the list of studies
  whose metadata could not be found.

Both directories are gitignored.

In [ ]:
inputs_dir = './inputs'
outputs_dir = './outputs'
studies_dir = os.path.join(inputs_dir, 'studies')

os.makedirs(inputs_dir, exist_ok=True)
os.makedirs(outputs_dir, exist_ok=True)
os.makedirs(studies_dir, exist_ok=True)

# Get Studies With Preharmonized Data

Authenticate against the commons, fetch the discovery metadata, and pick the studies to process.

In [ ]:
auth = Gen3Auth(commons_url, refresh_file=credentials_path)
mds = Gen3Metadata(commons_url, auth_provider=auth)
file_client = Gen3File(auth)

In [ ]:
studies = get_active_preharmonized_studies(mds)
print(f'Retrieved {len(studies)} active studies with pre-harmonized data')

In [ ]:
studies_to_process = select_studies(
    studies,
    mode=download_mode,
    selected_ids=selected_study_ids,
    max_count=max_studies_to_process,
)
print(f'Processing {len(studies_to_process)} of {len(studies)} studies (mode: {download_mode})')

studies_file = os.path.join(outputs_dir, 'preharmonized_studies.json')
with open(studies_file, 'w', encoding='utf-8') as f:
    json.dump(dict(studies_to_process), f, indent=4, ensure_ascii=False)
print(f'Saved study index to {studies_file}')

# Download Preharmonized PFBs

Download the PFB AVRO archive for each study and convert it to TSV manifests. Studies whose
files are already on disk are skipped, so re-running only fetches what is missing.

In [ ]:
print(f'Fetching preharmonized PFBs and extracting TSVs for {len(studies_to_process)} studies...')

for study_id, study_metadata in studies_to_process:
    study_dir = os.path.join(studies_dir, study_id)
    print(f'\n[{study_id}]')
    if download_pfb_for_study(study_metadata, study_dir, file_client):
        # 'tsvs' is the gen3 CLI's own default output directory, and the name
        # the metadata step below looks for.
        convert_pfb_to_tsv(study_dir, output_dir='tsvs')

print('\nAll done!')

# Download Metadata Files

Read the DRS URIs out of the TSV manifests and download the dbGaP data dictionary and variable
report XMLs. Files bundled in archives (`.tar.gz`, `.zip`) are extracted automatically, and
anything that fails is recorded in `failed_downloads.json` inside the study directory.

In [ ]:
print(f'Downloading metadata files for {len(studies_to_process)} studies...')

session = requests.Session()

for study_id, _ in studies_to_process:
    study_dir = os.path.join(studies_dir, study_id)
    print(f'\n[{study_id}]')
    download_study_metadata(study_dir, file_client, session)

print('\nAll done!')

# Check for Missing Metadata

Some studies have no data dictionary or variable report published at all. Those are listed here
and written to `outputs/` so they can be investigated separately — the next notebook silently
produces nothing for them.

In [ ]:
missing_metadata = check_missing_metadata(studies_to_process, studies_dir)

for entry in missing_metadata:
    print(f'[!] {entry["study_id"]}: missing_data_dict={entry["missing_data_dict"]}, missing_var_report={entry["missing_var_report"]}')

print(f'\n{len(missing_metadata)} of {len(studies_to_process)} studies have missing metadata files.')

if missing_metadata:
    missing_path = os.path.join(outputs_dir, 'studies_missing_metadata.json')
    with open(missing_path, 'w', encoding='utf-8') as f:
        json.dump(missing_metadata, f, indent=4)
    print(f'Saved to {missing_path}')
else:
    print('\nReady for the next step: harmonize_studies.ipynb')